# Setup

## Config

In [1]:
from pprint import pprint
import os
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import optuna
from transformers import set_seed
from transformers.utils import logging
from kaggle.api.kaggle_api_extended import KaggleApi
from huggingface_hub.utils import enable_progress_bars
from transformers.utils import logging

# Seed
seed = 42
set_seed(seed)

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
use_bf16 = (device.type == 'cuda') and (torch.cuda.is_bf16_supported())

# Kaggle auth
api = KaggleApi()
api.authenticate()

# Suppress loggings
os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'
logging.set_verbosity_error()
logging.disable_progress_bar()

# HF progress bar
enable_progress_bars()
logging.set_verbosity_info()

c:\Users\yana\Desktop\Library\AI\ai-competitions\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1010 21:24:03.978000 28884 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


## Parameters

In [2]:
data_path   = './data'
train_path  = 'data/train.csv'
test_path   = 'data/test.csv'

competition_name = 'llm-classification-finetuning'

# 1. Data

In [3]:
from pathlib import Path
import kagglehub
from datasets import load_dataset, DatasetDict, ClassLabel

# Download
if not (Path(train_path).is_file() and Path(test_path).is_file()):
    kagglehub.competition_download(
        handle=competition_name,
        output_dir=data_path,
    )
else:
    print("Competition data already exists.")

# Load
train_ds = load_dataset('csv', data_files=train_path)['train']
test_ds  = load_dataset('csv', data_files=test_path)['train']

# Features
cols_feature = ['prompt', 'response_a', 'response_b']
X_train = train_ds.select_columns(cols_feature)
X_test  = test_ds.select_columns(cols_feature)

# Labels
cols_label = ['winner_model_a', 'winner_model_b', 'winner_tie']
y_train = train_ds.select_columns(cols_label)

# Split
label = np.column_stack(
    (
        y_train['winner_model_a'],
        y_train['winner_model_b'],
        y_train['winner_tie'],
    )
).argmax(axis=1)
train_ds = train_ds.add_column('label', label)
train_ds = train_ds.cast_column(
    'label',
    ClassLabel(names=['win_a', 'win_b', 'tie'])
)

split = train_ds.train_test_split(
    test_size=0.2,
    stratify_by_column='label',
    seed=seed,
)

ds = DatasetDict({
    'train': split['train'],
    'valid': split['test'],
    'test': test_ds,
})

Competition data already exists.


In [4]:
print(f'n_train: {len(ds['train'])}')
print(f'n_valid: {len(ds['valid'])}')
print(f'n_test: {len(ds['test'])}')

n_train: 45981
n_valid: 11496
n_test: 3


In [5]:
idx = 0
with open('samples.txt', 'w', encoding='utf-8') as f:
    sample = X_train[idx]
    pprint(sample, stream=f)
    pprint(f'label: {y_train[idx]}', stream=f)

# 2. Baseline

## 1) Tokenizer

In [6]:
train_ds[0]['prompt'][:100]

'["Is it morally right to try to have a certain percentage of females on managerial positions?","OK, '

In [7]:
from transformers import AutoTokenizer

base_model_name = "google/gemma-2-2b-it"

tokenizer = AutoTokenizer.from_pretrained(base_model_name)

def build_input(prompt, resp_a, resp_b):
    return f"""
Choose which response a human prefers:
[Prompt] {prompt}
[Response_A] {resp_a}
[Response_B] {resp_b}
"""

def tokenize(batch):
    prompt = batch['prompt'][:256]
    resp_a = batch['response_a'][:512]
    resp_b = batch['response_b'][:512]

    inputs = [
        build_input(prompt[i], resp_a[i], resp_b[i])
        for i in range(len(prompt))
    ]

    return tokenizer(
        inputs,
        truncation=True,
        max_length=1_300,  # Speed/memory limit; long inputs may lose response B.
    )

train_tokenized = ds['train'].map(tokenize, batched=True, batch_size=128)
valid_tokenized = ds['valid'].map(tokenize, batched=True, batch_size=128)
test_tokenized  = ds['test'].map(tokenize, batched=True, batch_size=128)

c:\Users\yana\Desktop\Library\AI\ai-competitions\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\yana\.cache\huggingface\hub\models--google--gemma-2-2b-it. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
[transformers] loading configuration file config.json from cache at C:\Users\yana\.cache\huggi

## 2) Data Collator

In [8]:
from transformers import DataCollatorWithPadding

collator = DataCollatorWithPadding(tokenizer=tokenizer, pad_to_multiple_of=8)

## 3) Model

In [9]:
from transformers import AutoModelForSequenceClassification

base_model = AutoModelForSequenceClassification.from_pretrained(
    base_model_name,
    num_labels=3,
    problem_type="single_label_classification",  # Class IDs: 0=win_a, 1=win_b, 2=tie.
    dtype=torch.bfloat16 if use_bf16 else torch.float32,
).to(device)

[transformers] loading configuration file config.json from cache at C:\Users\yana\.cache\huggingface\hub\models--google--gemma-2-2b-it\snapshots\299a8560bedf22ed1c72a8a11e7dce4a7f9f51f8\config.json
[transformers] Model config Gemma2Config {
  "architectures": [
    "Gemma2ForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "attn_logit_softcapping": 50.0,
  "bos_token_id": 2,
  "cache_implementation": "hybrid",
  "dtype": "bfloat16",
  "eos_token_id": [
    1,
    107
  ],
  "final_logit_softcapping": 30.0,
  "head_dim": 256,
  "hidden_act": "gelu_pytorch_tanh",
  "hidden_activation": "gelu_pytorch_tanh",
  "hidden_size": 2304,
  "id2label": {
    "0": "LABEL_0",
    "1": "LABEL_1",
    "2": "LABEL_2"
  },
  "initializer_range": 0.02,
  "intermediate_size": 9216,
  "label2id": {
    "LABEL_0": 0,
    "LABEL_1": 1,
    "LABEL_2": 2
  },
  "layer_types": [
    "sliding_attention",
    "full_attention",
    "sliding_attention",
    "full_attention",
    "sliding_atte

## 3) Fine Tuning (Head-only)

In [ ]:
from transformers import Trainer, TrainingArguments, EarlyStoppingCallback
from scipy.special import softmax
from sklearn.metrics import log_loss, accuracy_score

# Freeze
for name, param in base_model.model.named_parameters():
    param.requires_grad = False

# Keep trainable head weights in FP32; Trainer still uses BF16 autocast.
base_model.score.float()

# compute_metrics
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = softmax(logits, axis=-1)
    preds = np.argmax(probs, axis=-1)

    return {
        "log_loss": log_loss(labels, probs, labels=[0, 1, 2]),
        "accuracy": accuracy_score(labels, preds),
    }

# TrainingArgs
training_args = TrainingArguments(
    output_dir='../tmp/outputs/arena_baseline',
    num_train_epochs=2,

    # Learning rate.
    learning_rate=3e-4,     # relatively large for head-only tuning.
    lr_scheduler_type='linear',
    warmup_steps=0.1,       # first 10% steps: 0 -> lr, remaining: lr -> 0.

    # Optimizer.
    weight_decay=1e-2,      # slightly decrease weights per update.
    max_grad_norm=1.0,      # clipping L2-norm of the gradient.

    # bf16.
    bf16=use_bf16,          # use bf16 instead of fp16. 
    
    # Batch size.
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    train_sampling_strategy="group_by_length",  # Reduce padding (Transformers 5.19).
    gradient_accumulation_steps=4,

    # Checkpoint.
    eval_strategy='epoch',      # run validation after every epoch.
    save_strategy='epoch',      # save a checkpoint.
    logging_strategy='steps',   # report training loss.
    logging_steps=100,          # log training loss every 100 steps.
    save_total_limit=2,         # keep only a small number of checkpoints.

    # Best model.
    load_best_model_at_end=True,    # restore the best model, instead keeping the last one.
    metric_for_best_model='log_loss',
    greater_is_better=False,

    # Progress.
    disable_tqdm=False,     # make sure tqdm is shown.    
)

# Early stopping
early_stopping = EarlyStoppingCallback(
    early_stopping_patience=2,      # tolerate up to 2 consecutives without improvements > threshold.
    early_stopping_threshold=1e-3,  # should improve at least this value.
)

# Trainer
trainer = Trainer(
    model=base_model,
    args=training_args,
    callbacks=[early_stopping],

    train_dataset=train_tokenized,
    eval_dataset=valid_tokenized,

    data_collator=collator,
    
    compute_metrics=compute_metrics,

    processing_class=tokenizer,     # saved together with model.
)

# Train
base_model.config.use_cache = False
training_args.use_cache = False
history = trainer.train()

# Evaluate
eval_results = trainer.evaluate()